In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [3]:
!pip install langchain-openai

In [ ]:
!pip install faiss-gpu-cu12

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
!pip install pymupdf

In [28]:
import os

# 여기에 실제 OpenAI API 키를 입력하세요
os.environ["OPENAI_API_KEY"] = ""

In [13]:
# 문서로드
loader = PyMuPDFLoader("paper1.pdf")
docs = loader.load()

# 문서분할(split documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 임베딩 생성
embeddings = OpenAIEmbeddings()

# DB생성
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

In [23]:
# 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색 및 생성
retriever =vectorstore.as_retriever()

In [24]:
retriever.invoke("최저임금 인상이 정규직과비정규직 임금격차에 미치는영향에 대한 논문저자는 누구야?")

[Document(id='dbe3b03b-0fba-4fe5-85d6-d1fa1eecf2a3', metadata={'producer': 'Hancom PDF 1.3.0.538', 'creator': 'Haansoft Hangul 2007 7, 5, 12, 710', 'creationdate': '2025-11-19T16:42:49+09:00', 'source': 'paper1.pdf', 'file_path': 'paper1.pdf', 'total_pages': 27, 'format': 'PDF 1.4', 'title': '', 'author': 'User', 'subject': '', 'keywords': '', 'moddate': '2025-11-19T16:42:49+09:00', 'trapped': '', 'modDate': "D:20251119164249+09'00'", 'creationDate': "D:20251119164249+09'00'", 'page': 4}, page_content='최저임금 인상이 정규직과 비정규직 임금격차에 미치는 영향(진민준)  \uf000\n99\n을 실시하였다. 이를 통해 사전 기간의 평행추세 가정을 검증하고, 정책 시행 이후 \n연도별로 정규직과 비정규직 간, 그리고 임금집단 간 임금격차의 동태적 변화를 \n추정하였다.\n특히 본 연구에서는 임금수준에 따라 전체 표본을 세 개의 집단으로 구분하여 \n분석을 수행하였다. 첫째, 최저임금 미만집단은 해당 연도의 법정 최저임금보다 \n낮은 임금을 받는 근로자로 정의하였다. 둘째, 최저임금 영향집단은 전년 대비 최\n저임금 인상으로 인해 직접적인 영향을 받았을 것으로 추정되는 구간의 임금 수혜\n자들로 구성하였다. 셋째, 차상위 임금집단은 최저임금보다 높은 수준의 임금을 \n받고 있어 직접적인 영향은 적지만 간접적인 파급효과를 받을 수 있는 근로자들을 \n포함하였다. 이러한 집단 구분 방식은 선행연구(양진보ㆍ김기승, 2024)를 참조하\n여 설정하였으며, 이를 통해 최저임금 인상

In [25]:
# 프롬프트 생성
prompt = PromptTemplate.from_template(
    """You are a specialized QA assistant.
    Answer the question using only the provided context.
    If the information is insufficient to provide an answer, state that you do not know.
    Please respond in Korean.

#Question:
{question}

#Context:
{context}

#Answer:"""
)

# LLM 생성
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

chain = (
    {"context":retriever, "question":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [26]:
# 체인 실행
question = "'최저임금 인상이 정규직과비정규직 임금격차에 미치는 영향'논문의 저자는 누구야?"
chain.invoke(question)

"'최저임금 인상이 정규직과 비정규직 임금격차에 미치는 영향' 논문의 저자는 진민준입니다."

In [27]:
question = "'최저임금 인상이 정규직과비정규직 임금격차에 미치는 영향'논문의 주제는 무엇인가요?"
chain.invoke(question)

"'최저임금 인상이 정규직과 비정규직 임금격차에 미치는 영향' 논문의 주제는 최저임금 인상이 정규직과 비정규직 간의 임금격차에 미치는 영향을 실증적으로 분석하는 것입니다. 이 연구는 한국노동패널(KLIPS) 자료를 활용하여 2017년부터 2020년까지의 기간 동안 최저임금 인상이 임금격차에 미친 영향을 분석하였으며, 이벤트 스터디와 삼중차분 모형을 사용하여 정책 시행 이전과 이후의 동태적 효과를 추정하였습니다. 연구 결과, 최저임금 인상이 단기적으로는 저임금층의 임금 개선에 기여하였으나, 시간이 경과함에 따라 고용형태 간 임금격차 완화 효과가 약화되었음을 시사합니다."